# Llama-3.2-1B × opc-sft-stage2 leaderboard — cross-model robustness cell (r=64, 9000 steps)

Cross-model cell from `~/.claude/plans/as-part-of-our-tender-quilt.md`. Same opc-sft-stage2 dataset, same horizon, same optimizer arms as Phase L — but base model swapped to Llama-3.2-1B to address the 'does this work on non-OLMo bases' reviewer objection without a full Phase B/C spend.

Cell: Llama-3.2-1B × opc-sft-stage2 (all 4 sub-configs, Llama-tokenized cache `data/opc_sft_stage2_all_packed_seq2048_llama32`) × global_batch=16 (batch=4 × accum=4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

Source label: **Llama OPC eval JSONL logs**: `meta-llama/Llama-3.2-1B`; `data/opc_sft_stage2_all_packed_seq2048_llama32`; `packed_v1.1`; seq2048; 9k-step eval logs.

Diagnostic q_agree source: **Llama OPC JSONL q_agree**: `optim_step.awc_q_agree_*` from `logs/chord_tight_slack_llama32_1b_opc_r256_lr2_blackwell` (r256, ns=5, lr={3e-3,1e-2}) and `logs/chord_tight_slack_llama32_1b_opc_r256_ns8_lr2_blackwell` (r256, ns=8, lr={3e-3,1e-2}). These are JSONL diagnostics, not snapshot-registry rows.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4}
- **chord-tight k=1** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2}

Source log groups: `{adamw,chord_tight}_robustness_llama32_1b_opc_r64_blackwell` (2 groups).

**Pass criterion** (per plan): Δ direction matches OLMo (tight-chord < AdamW) at any of the 3 LRs. Magnitude can differ; direction cannot. If direction inverts, the result is OLMo-specific and must be reported as such.

**σ anchor**: borrowed `σ_AdamW(packed_v1, opc-sft-stage2, OLMo, r=64) = 0.0017` until Llama multi-seed exists. Treat σ-units as a rough scale only.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Repo root by marker-walk, so this notebook works from any subdir.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lora_playground').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.plotting import leaderboard_panel, canonical_label

# Run membership comes from the shared registry (lora_playground.workloads) — the
# SAME source the leaderboard doc uses, so notebook and doc cannot drift. Cells are
# leaderboard_panel(model, dataset, rank, ...) + an optional label_filter(label, cfg).

def ns_of(cfg):
    oc = cfg.get('optimizer_config') or {}
    return cfg.get('muon_ns_steps', oc.get('ns_steps'))

def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def is_curv(label):
    # curvature-whitening / SOAP-curv / KL-Shampoo arms.
    return ('SOAP-curv' in label) or ('KL-Shampoo' in label) or ('+curv' in label)

## r=64

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 64,
    'Llama-3.2-1B × opc r=64',
    figsize=(11, 4))
plt.show()
sdf

## r=256 (rank-extension robustness)

Same two arms, r=256. Source groups: `{adamw,chord_tight}_robustness_llama32_1b_opc_r256_blackwell`.

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 256,
    'Llama-3.2-1B × opc r=256',
    figsize=(11, 4))
plt.show()
sdf

## NS-iteration (whitening-fraction) sweep \u2014 does Llama's optimal whitening differ from OLMo's NS-5?

chord-tight with `muon_ns_steps` \u2208 {3, 5, 8} \u2192 whitening fraction \u2248 {0.47, 0.72, 1.0} (measured on the Llama gradient spectrum, stable_rank 10.2). j=5 is the base group. AdamW shown for reference. If a non-5 j beats ns=5, Llama's optimal whitening differs from the OLMo-tuned default.

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 64,
    'Llama-3.2-1B × opc r=64 — NS-iteration (whitening) sweep',
    label_filter=lambda l, c: l == 'AdamW' or (picard_of(c) == 1 and 'clean' not in l and not is_curv(l)),
    figsize=(11, 4))
plt.show()
sdf

## Picard ablation \u2014 does the 1/\u03b7 cross-coupling add value on Llama?

chord-tight-CLEAN, polar_method=ns, muon_ns_steps=8 (full whitening). picard=1 collapses to plain polar (cross-coupling term identically zero); picard=2 activates the 1/\u03b7 coupling. If picard=2 beats picard=1 at best-lr, the coupling adds value on Llama (as it did on OLMo packed_v1). AdamW shown for reference.

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 64,
    'Llama-3.2-1B × opc r=64 — picard ablation (1/η cross-coupling)',
    label_filter=lambda l, c: l == 'AdamW' or ('ns=8' in l and not is_curv(l)),
    figsize=(11, 4))
plt.show()
sdf

## r=256 — NS-iteration (whitening-fraction) sweep

chord-tight at r=256 with `muon_ns_steps` ∈ {5, 8}. ns=5 is the base `chord-tight k=1` group; ns=8 (`chord_tight_robustness_llama32_1b_opc_r256_ns8_blackwell`) is the full-whitening rank-extension. AdamW shown for reference. (No ns=3 arm at r=256.)

Diagnostic source note: `q_agree`, `chord_slack`, stable-rank, saturation, and conditioning values discussed for Llama come from JSONL `optim_step` events in `logs/chord_tight_slack_llama32_1b_opc_r256_lr2_blackwell` and `logs/chord_tight_slack_llama32_1b_opc_r256_ns8_lr2_blackwell`. Those are Llama OPC packed-v1.1 runs using `data/opc_sft_stage2_all_packed_seq2048_llama32`; they are not snapshot-registry rows.


In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 256,
    'Llama-3.2-1B × opc r=256 — NS-iteration (whitening) sweep',
    label_filter=lambda l, c: l == 'AdamW' or (picard_of(c) == 1 and 'clean' not in l and not is_curv(l)),
    figsize=(11, 4))
plt.show()
sdf

## r=256 — Picard ablation (does the 1/η cross-coupling add value?)

Same picard ablation as r=64, at r=256. `ns8 picard=1 (ref, non-clean)` = `chord_tight_robustness_llama32_1b_opc_r256_ns8_blackwell` (cross-coupling identically zero); `clean ns8 picard=2` = `chord_tight_clean_ns8_picard_ablation_llama32_1b_opc_r256_blackwell` (1/η coupling active). Both arms share lr ∈ {3e-4, 1e-3, 3e-3, 1e-2}. AdamW shown for reference.

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('meta-llama/Llama-3.2-1B', 'opc', 256,
    'Llama-3.2-1B × opc r=256 — picard ablation (1/η cross-coupling)',
    label_filter=lambda l, c: l == 'AdamW' or ('ns=8' in l and not is_curv(l)),
    figsize=(11, 4))
plt.show()
sdf